# **Data Loading**

In [1]:
# Import the required libraries
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("transport_raw.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDataset Overview:", df.head())

Shape: (1335, 15)

Columns: ['Route ID', 'Route Name', 'Bus Number', 'Bus Type', 'Date', 'Departure Time', 'Arrival Time', 'Delay Duration', 'Passenger Count', 'Ticket Revenue', 'Fuel Consumption', 'Driver ID', 'Weather Condition', 'Day Type', 'Trip Status']

Dataset Overview:   Route ID                          Route Name Bus Number  Bus Type  \
0      R08  South Terminal - Business District    BUS-309  Standard   
1      R05               West Gate - Tech Park    BUS-280  Standard   
2      R02          North Station - University    BUS-109        AC   
3      R04            East Suburb - Mall Plaza    BUS-392        AC   
4      R03                   Harbor - Downtown    BUS-367      MINI   

         Date Departure Time Arrival Time  Delay Duration  Passenger Count  \
0  2025-03-16          16:30        17:13               5              NaN   
1  2025-02-18          12:15        13:34               8             43.0   
2  2025-01-17          19:13        18:30               7    

# **Data Exploration**

In [2]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATES ===")
print("Duplicate rows:", df.duplicated().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== BUS TYPE VARIANTS ===")
print(df["Bus Type"].value_counts(dropna=False))

print("\n=== WEATHER VARIANTS ===")
print(df["Weather Condition"].value_counts(dropna=False))

print("\n=== DAY TYPE VARIANTS ===")
print(df["Day Type"].value_counts(dropna=False))

=== MISSING VALUES ===
Route ID              0
Route Name            0
Bus Number            0
Bus Type              0
Date                  0
Departure Time        0
Arrival Time          0
Delay Duration        0
Passenger Count      52
Ticket Revenue        0
Fuel Consumption     40
Driver ID            35
Weather Condition    68
Day Type              0
Trip Status           0
dtype: int64

=== DUPLICATES ===
Duplicate rows: 35

=== DATA TYPES ===
Route ID              object
Route Name            object
Bus Number            object
Bus Type              object
Date                  object
Departure Time        object
Arrival Time          object
Delay Duration         int64
Passenger Count      float64
Ticket Revenue        object
Fuel Consumption      object
Driver ID             object
Weather Condition     object
Day Type              object
Trip Status           object
dtype: object

=== BUS TYPE VARIANTS ===
Bus Type
Standard           518
AC                 379
Electric      

# **Data Cleaning**

# Removing Duplicates:

In [3]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows. {before} → {len(df)}")

Removed 35 duplicate rows. 1335 → 1300


# Standardization:

In [4]:
# Bus Type: map all 15 variants to 4 clean values
bus_map = {
    "AC":"AC","A/C":"AC","a.c.":"AC","Air Conditioned":"AC","ac":"AC",
    "Standard":"Standard","STD":"Standard","Std":"Standard","standard":"Standard",
    "Electric":"Electric","EV":"Electric","electric":"Electric",
    "Mini":"Mini","MINI":"Mini","mini":"Mini",
}
df["Bus Type"] = df["Bus Type"].str.strip().replace(bus_map)
print("Bus Type after:", sorted(df["Bus Type"].dropna().unique()))

# Weather: map variants to 5 clean values
weather_map = {
    "Clear":"Clear","clear":"Clear","CLEAR":"Clear",
    "Rain":"Rain","rain":"Rain","Rainy":"Rain",
    "Fog":"Fog","fog":"Fog","Foggy":"Fog",
    "Cloudy":"Cloudy","cloudy":"Cloudy","overcast":"Cloudy",
    "Storm":"Storm","storm":"Storm","Stormy":"Storm",
}
df["Weather Condition"] = df["Weather Condition"].str.strip().replace(weather_map)
print("Weather after:", sorted(df["Weather Condition"].dropna().unique()))

# Day Type: strip whitespace and title-case
df["Day Type"] = df["Day Type"].str.strip().str.title()
print("Day Type after:", sorted(df["Day Type"].unique()))

Bus Type after: ['AC', 'Electric', 'Mini', 'Standard']
Weather after: ['Clear', 'Cloudy', 'Fog', 'Rain', 'Storm']
Day Type after: ['Weekday', 'Weekend']


# Fixing Time Format:

In [5]:
def to_24h(t):
    if pd.isna(t): return t
    t = str(t).strip()
    for fmt in ("%H:%M", "%I:%M %p"):
        try:
            return pd.to_datetime(t, format=fmt).strftime("%H:%M")
        except:
            continue
    return np.nan  # unparseable

df["Departure Time"] = df["Departure Time"].apply(to_24h)
df["Arrival Time"] = df["Arrival Time"].apply(to_24h)
print(df[["Departure Time","Arrival Time"]].head())

  Departure Time Arrival Time
0          16:30        17:13
1          12:15        13:34
2          19:13        18:30
3          16:45        17:54
4          19:00        20:42


# Fixing Date Column:

In [6]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True, errors="coerce")
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Unparseable dates:", df["Date"].isna().sum())

Date range: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Unparseable dates: 0


# Data Type Correction:

In [7]:
# Ticket Revenue: remove "Rs", commas, spaces
df["Ticket Revenue"] = (
    df["Ticket Revenue"].astype(str)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["Ticket Revenue"] = pd.to_numeric(df["Ticket Revenue"], errors="coerce")

# Fuel Consumption: remove " L"
df["Fuel Consumption"] = (
    df["Fuel Consumption"].astype(str)
    .str.replace("L", "", regex=False)
    .str.strip()
)
df["Fuel Consumption"] = pd.to_numeric(df["Fuel Consumption"], errors="coerce")

# Passenger Count and Delay Duration to numeric too
df["Passenger Count"] = pd.to_numeric(df["Passenger Count"], errors="coerce")
df["Delay Duration"] = pd.to_numeric(df["Delay Duration"], errors="coerce")

print(df.dtypes[["Ticket Revenue","Fuel Consumption","Passenger Count","Delay Duration"]])

Ticket Revenue      float64
Fuel Consumption    float64
Passenger Count     float64
Delay Duration        int64
dtype: object


# Data Validation:

In [8]:
# Build full datetime for departure/arrival to compare them
dep_dt = pd.to_datetime(df["Date"].astype(str) + " " + df["Departure Time"], errors="coerce")
arr_dt = pd.to_datetime(df["Date"].astype(str) + " " + df["Arrival Time"], errors="coerce")

# Rows where arrival is before departure (the swapped ones)
invalid = arr_dt < dep_dt
print("Arrival-before-departure rows:", invalid.sum())

# Fix by swapping them back
df.loc[invalid, ["Departure Time","Arrival Time"]] = df.loc[invalid, ["Arrival Time","Departure Time"]].values
print("Swapped back.")

# Negative delays are impossible — set to NaN (will handle as missing)
neg_delay = df["Delay Duration"] < 0
print("Negative delay rows:", neg_delay.sum())
df.loc[neg_delay, "Delay Duration"] = np.nan

Arrival-before-departure rows: 15
Swapped back.
Negative delay rows: 10


# Outlier Detection And Handling

In [9]:
# Bus capacity per type
capacity = {"Standard":50,"AC":45,"Electric":40,"Mini":25}
df["capacity"] = df["Bus Type"].map(capacity)

# Passenger count: negative or over capacity = impossible
bad_pax = (df["Passenger Count"] < 0) | (df["Passenger Count"] > df["capacity"])
print("Impossible passenger counts:", bad_pax.sum())
df.loc[bad_pax, "Passenger Count"] = np.nan

# Delay: cap absurd values. Use a sensible threshold (e.g. > 120 min is implausible)
bad_delay = df["Delay Duration"] > 120
print("Absurd delays (>120 min):", bad_delay.sum())
df.loc[bad_delay, "Delay Duration"] = np.nan

# Fuel: flag extreme values with the IQR method
Q1, Q3 = df["Fuel Consumption"].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper = Q3 + 3*IQR   # 3×IQR = extreme outliers only
bad_fuel = df["Fuel Consumption"] > upper
print(f"Extreme fuel outliers (>{upper:.1f} L):", bad_fuel.sum())
df.loc[bad_fuel, "Fuel Consumption"] = np.nan

df = df.drop(columns=["capacity"])

Impossible passenger counts: 6
Absurd delays (>120 min): 2
Extreme fuel outliers (>27.4 L): 4


# Handling Missing Values

In [10]:
# Numeric: median (robust). Passenger Count and Delay use median.
df["Passenger Count"] = df["Passenger Count"].fillna(df["Passenger Count"].median()).round().astype(int)
df["Delay Duration"] = df["Delay Duration"].fillna(df["Delay Duration"].median()).round().astype(int)

# Fuel: median within Bus Type (electric ≈ 0, others higher — category matters)
df["Fuel Consumption"] = df.groupby("Bus Type")["Fuel Consumption"].transform(
    lambda s: s.fillna(s.median())
).round(2)

# Categorical: fill with mode (most common) or "Unknown"
df["Weather Condition"] = df["Weather Condition"].fillna(df["Weather Condition"].mode()[0])
df["Driver ID"] = df["Driver ID"].fillna("Unknown")

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


# Remaining Fixes:

In [13]:
# Re-check and fix any remaining over-capacity passenger counts
capacity = {"Standard":50, "AC":45, "Electric":40, "Mini":25}
df["cap"] = df["Bus Type"].map(capacity)

over = df["Passenger Count"] > df["cap"]
print(f"Over-capacity rows found: {over.sum()}")
print(df.loc[over, ["Bus Type","Passenger Count","cap"]])

# Cap them at the bus's capacity (a defensible fix — the bus can't hold more than its seats)
df.loc[over, "Passenger Count"] = df.loc[over, "cap"]

df = df.drop(columns=["cap"])
print("Fixed. Over-capacity now:", (df["Passenger Count"] > df["Bus Type"].map(capacity)).sum())

Over-capacity rows found: 6
     Bus Type  Passenger Count  cap
136      Mini               33   25
169      Mini               33   25
898      Mini               33   25
903      Mini               33   25
1113     Mini               33   25
1319     Mini               33   25
Fixed. Over-capacity now: 0


# **Final Verification**

In [14]:
print("=== FINAL CHECK ===")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Bus Types:", sorted(df["Bus Type"].unique()))
print("Weather:", sorted(df["Weather Condition"].unique()))
print("Day Types:", sorted(df["Day Type"].unique()))
print("Revenue is numeric:", pd.api.types.is_numeric_dtype(df["Ticket Revenue"]))
print("Fuel is numeric:", pd.api.types.is_numeric_dtype(df["Fuel Consumption"]))
print("Negative delays:", (df["Delay Duration"] < 0).sum())

=== FINAL CHECK ===
Shape: (1300, 15)
Missing values: 0
Duplicates: 0
Bus Types: ['AC', 'Electric', 'Mini', 'Standard']
Weather: ['Clear', 'Cloudy', 'Fog', 'Rain', 'Storm']
Day Types: ['Weekday', 'Weekend']
Revenue is numeric: True
Fuel is numeric: True
Negative delays: 0


# **Cleaned Dataset Export**

In [15]:
df.to_csv("transport_cleaned.csv", index=False)
print("\nSaved transport_cleaned.csv")


Saved transport_cleaned.csv
